# Create correct format CSV

In [1]:
# # Cell 1: build mapping with one column per extractor
# import pandas as pd
# from pathlib import Path

# # 1. load your base CSV
# df = pd.read_csv(
#     r"C:\Users\Dhyey\Documents\University\MSc\CS907 Dissertation Project\Tutorials\Data\CAMELYON\class_mapping.csv"
# )  # has columns sample_id,label,set

# # 2. list all extractors (must match your folders under feats/)
# extractors = [
#     "conch_v1","conch_v15","ctranspath","gigapath","hibou_l",
#     "hoptimus0","hoptimus1","midnight12k","phikon","phikon_v2",
#     "resnet50","uni_v1","uni_v2","virchow","virchow2"
# ]

# root_dir = Path(
#     r"C:\Users\Dhyey\Documents\University\MSc\CS907 Dissertation Project\Tutorials\Data\CAMELYON"
# )

# # 3. add one path_<ext> column per extractor
# for ext in extractors:
#     col = f"path_{ext}"
#     df[col] = df.apply(
#         lambda r: str(
#             root_dir
#             / r["set"]                  # train or test
#             / "feats"
#             / f"features_{ext}"
#             / f"{r['sample_id']}.h5"
#         ),
#         axis=1
#     )

# # 4. (optional) save for inspection
# df.to_csv(r"data\CAMELYON\mapping_multiextractor.csv", index=False)
# print("Added columns:", [f"path_{e}" for e in extractors], "\nTotal rows:", len(df))

# CLAM

In [2]:
# Cell 1: Imports & CLAM_SB architecture

import os
from pathlib import Path
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix
)

# ─── Attention Networks ────────────────────────────────────────
class Attn_Net_Gated(nn.Module):
    """Gated attention: two parallel linear heads (Tanh & Sigmoid) then combine."""
    def __init__(self, L, D, dropout=False, n_classes=1):
        super().__init__()
        self.attention_a = nn.Sequential(
            nn.Linear(L, D), nn.Tanh(),
            *( [nn.Dropout(0.25)] if dropout else [] )
        )
        self.attention_b = nn.Sequential(
            nn.Linear(L, D), nn.Sigmoid(),
            *( [nn.Dropout(0.25)] if dropout else [] )
        )
        self.attention_c = nn.Linear(D, n_classes)

    def forward(self, x):
        a = self.attention_a(x)
        b = self.attention_b(x)
        A = a.mul(b)  # elementwise gating
        return self.attention_c(A), x  # (attention_logits, features)

# ─── Single‐Branch CLAM ────────────────────────────────────────
class CLAM_SB(nn.Module):
    def __init__(
        self,
        embed_dim=1024,
        n_classes=2,
        gate=True,
        size_arg="small",
        dropout=0.0,
        k_sample=8,
        instance_loss_fn=nn.CrossEntropyLoss(),
        subtyping=False,
    ):
        super().__init__()
        size_map = {"small": [embed_dim, 512, 256], "big": [embed_dim, 512, 384]}
        size = size_map[size_arg]

        # feature → attention layers
        self.attention_net = nn.Sequential(
            nn.Linear(size[0], size[1]), nn.ReLU(), nn.Dropout(dropout),
            (Attn_Net_Gated if gate else Attn_Net)(L=size[1], D=size[2], dropout=dropout, n_classes=1)
        )

        # bag‐level classifier
        self.classifiers = nn.Linear(size[1], n_classes)

        # instance‐level classifiers (top-k supervision, optional)
        self.instance_classifiers = nn.ModuleList([
            nn.Linear(size[1], 2) for _ in range(n_classes)
        ])

        self.k_sample = k_sample
        self.instance_loss_fn = instance_loss_fn
        self.n_classes = n_classes
        self.subtyping = subtyping

    def forward(self, h):
        # h: [N_patches, embed_dim]
        A_logits, h_feats = self.attention_net(h)  # (N×1, N×D)
        A = A_logits.view(1, -1)                   # [1, N]
        A_norm = F.softmax(A, dim=1)               # normalize over patches

        # bag embedding
        M = torch.mm(A_norm, h_feats)              # [1, D]
        logits = self.classifiers(M)               # [1, n_classes]
        return logits, F.softmax(logits, dim=1)


In [3]:
# Cell 3: Split & DataLoaders with variable-size bag support and file existence check

import os
import pandas as pd
import h5py
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# Load the mapping with multi-extractor paths
df = pd.read_csv(os.path.join("data", "CAMELYON", "mapping_multiextractor.csv"))

# Choose which extractor’s column to use
extractor = "resnet50"
path_col  = f"path_{extractor}"

# Filter to only those where the .h5 feature file actually exists
df = df[df[path_col].apply(os.path.exists)].reset_index(drop=True)
print(f"After filtering: {len(df)} WSI bags with extracted features found for {extractor}.")

# 1) train / test split by the 'set' column
train_df = df[df["set"] == "train"].reset_index(drop=True)
test_df  = df[df["set"] == "test"].reset_index(drop=True)

# 2) carve out a stratified validation set from train
train_df, val_df = train_test_split(
    train_df,
    test_size=0.3,
    stratify=train_df["label"],
    random_state=42
)

print(f"Splits: {len(train_df)} train / {len(val_df)} val / {len(test_df)} test")

# 3) Dataset class returning (x, y) where x is [N_patches, D], y is 1‐element tensor
class H5BagDataset(Dataset):
    def __init__(self, df, path_col):
        self.df = df.reset_index(drop=True)
        self.path_col = path_col
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        with h5py.File(row[self.path_col], "r") as f:
            feats = f["features"][()]
        x = torch.from_numpy(feats).float()
        y = torch.tensor([row["label"]], dtype=torch.long)
        return x, y

# 4) collate_fn to avoid stacking variable-length x
def bag_collate_fn(batch):
    return batch[0]   # batch is [(x,y)], so return the single (x,y)

# 5) DataLoaders with batch_size=1
bs = 1
train_ds = H5BagDataset(train_df, path_col)
val_ds   = H5BagDataset(val_df,   path_col)
test_ds  = H5BagDataset(test_df,  path_col)

train_loader = DataLoader(
    train_ds, batch_size=bs, shuffle=True,  collate_fn=bag_collate_fn
)
val_loader   = DataLoader(
    val_ds,   batch_size=bs, shuffle=False, collate_fn=bag_collate_fn
)
test_loader  = DataLoader(
    test_ds,  batch_size=bs, shuffle=False, collate_fn=bag_collate_fn
)

print(f"Loaders ready: {len(train_ds)} train / {len(val_ds)} val / {len(test_ds)} test")

After filtering: 359 WSI bags with extracted features found for resnet50.
Splits: 161 train / 69 val / 129 test
Loaders ready: 161 train / 69 val / 129 test


In [4]:
# Cell 4: Model, optimizer & loss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CLAM_SB(
    embed_dim=1024,    # your patch embedding size
    n_classes=2,
    dropout=0.3
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-5)
loss_fn   = nn.CrossEntropyLoss()

In [5]:
# Cell 5: Training loop (loss and logics fixed)

n_epochs = 1000
best_val_auc = 0.0

for epoch in range(1, n_epochs+1):
    model.train()
    running_loss = 0.0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)   # x: [N_patches, D], y: [1]
        optimizer.zero_grad()
        logits, _ = model(x)                # logits: [1, 2]
        # now both input and target have batch size = 1
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    train_loss = running_loss / len(train_loader.dataset)
    
    # validation
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            logits, probs = model(x)
            all_probs.append(probs[:,1].cpu().numpy())
            all_labels.append(y.cpu().numpy())
    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    val_auc = roc_auc_score(all_labels, all_probs)

    print(f"Epoch {epoch}/{n_epochs}  train_loss={train_loss:.4f}  val_auc={val_auc:.4f}")

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model.state_dict(), "best_clam_sb.pt")
    
    print(f"Best ROCAUC: {best_val_auc}")

Epoch 1/1000  train_loss=0.6929  val_auc=0.6035
Best ROCAUC: 0.6035353535353536
Epoch 2/1000  train_loss=0.6922  val_auc=0.6296
Best ROCAUC: 0.6296296296296297
Epoch 3/1000  train_loss=0.6911  val_auc=0.6254
Best ROCAUC: 0.6296296296296297
Epoch 4/1000  train_loss=0.6903  val_auc=0.6246
Best ROCAUC: 0.6296296296296297
Epoch 5/1000  train_loss=0.6897  val_auc=0.6153
Best ROCAUC: 0.6296296296296297
Epoch 6/1000  train_loss=0.6887  val_auc=0.6195
Best ROCAUC: 0.6296296296296297
Epoch 7/1000  train_loss=0.6876  val_auc=0.6178
Best ROCAUC: 0.6296296296296297
Epoch 8/1000  train_loss=0.6867  val_auc=0.6162
Best ROCAUC: 0.6296296296296297
Epoch 9/1000  train_loss=0.6868  val_auc=0.6136
Best ROCAUC: 0.6296296296296297
Epoch 10/1000  train_loss=0.6861  val_auc=0.6128
Best ROCAUC: 0.6296296296296297
Epoch 11/1000  train_loss=0.6850  val_auc=0.6153
Best ROCAUC: 0.6296296296296297
Epoch 12/1000  train_loss=0.6847  val_auc=0.6120
Best ROCAUC: 0.6296296296296297
Epoch 13/1000  train_loss=0.6837  val

In [6]:
# Cell 6: Test evaluation

model.load_state_dict(torch.load("best_clam_sb.pt"))
model.eval()

y_true, y_prob = [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        _, probs = model(x)
        y_prob.extend(probs[:,1].cpu().numpy())
        y_true.extend(y.numpy())

y_true = np.array(y_true)
y_prob = np.array(y_prob)
thresh = 0.5
y_pred = (y_prob >= thresh).astype(int)

print("Test AUC:       ", roc_auc_score(y_true, y_prob))
print("Accuracy:       ", accuracy_score(y_true, y_pred))
print("Precision:      ", precision_score(y_true, y_pred))
print("Recall:         ", recall_score(y_true, y_pred))
print("F1 score:       ", f1_score(y_true, y_pred))
print("Confusion mat:\n", confusion_matrix(y_true, y_pred))

Test AUC:        0.6658163265306123
Accuracy:        0.5891472868217055
Precision:       0.4714285714285714
Recall:          0.673469387755102
F1 score:        0.5546218487394958
Confusion mat:
 [[43 37]
 [16 33]]
